# Sql Window Interview Problems

Run all cells in a fresh Python3.11 kernel. PostgreSQL runs only in a new temporary directory over a local Unix socket; no API keys, Docker, existing database or administrator privileges. The final regression cell closes the owned connection/server and removes only its temporary data. Original source code and output tables remain in nonexecuting appendices.

In [ ]:
import sys
import subprocess
from pathlib import Path
from tempfile import mkdtemp
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'pgserver==0.1.4', 'psycopg[binary]==3.2.12'], check=True)
import pgserver
import psycopg
from IPython.core.magic import register_cell_magic

data_dir = Path(mkdtemp(prefix='tdb-sql-interview-'))
server = pgserver.get_server(data_dir, cleanup_mode='delete')
connection = psycopg.connect(server.get_uri(), autocommit=True)
assert connection.execute('SHOW listen_addresses').fetchone()[0] == ''
query_results, query_texts, query_history = {}, {}, {}

def print_result(columns, rows):
    values = [['NULL' if value is None else str(value) for value in row] for row in rows]
    widths = [max([len(column)] + [len(row[i]) for row in values]) for i, column in enumerate(columns)]
    def line(row):
        return ' | '.join(value.ljust(width) for value, width in zip(row, widths)).rstrip()
    print(line(columns))
    print('-+-'.join('-' * width for width in widths))
    for row in values:
        print(line(row))
    print('Rows:', len(rows))

@register_cell_magic
def sql(line, cell):
    key = line.strip()
    query_texts[key] = cell.strip()
    history = []
    with connection.cursor() as cursor:
        cursor.execute(cell)
        while True:
            if cursor.description:
                columns, rows = [column.name for column in cursor.description], cursor.fetchall()
                print_result(columns, rows)
                query_results[key] = (columns, rows)
                history.append((columns, rows))
            else:
                print(cursor.statusmessage)
                history.append(cursor.statusmessage)
            if not cursor.nextset():
                break
    query_history[key] = history

print('Python:', sys.version.split()[0], '| PostgreSQL:', connection.execute('SHOW server_version').fetchone()[0])
print('Isolated temporary database; local Unix socket only. SQL magic ready.')

connection.execute("CREATE SCHEMA team_size;\nCREATE TABLE team_size.Employee(employee_id INTEGER PRIMARY KEY,team_id INTEGER);\nINSERT INTO team_size.Employee VALUES(1,8),(2,8),(3,8),(4,7),(5,9),(6,9);\nCREATE SCHEMA salary_ranking;\nCREATE TABLE salary_ranking.Department(id INTEGER PRIMARY KEY,name TEXT);\nINSERT INTO salary_ranking.Department VALUES(1,'IT'),(2,'Sales');\nCREATE TABLE salary_ranking.Employee(id INTEGER PRIMARY KEY,name TEXT,salary INTEGER,departmentId INTEGER REFERENCES salary_ranking.Department(id));\nINSERT INTO salary_ranking.Employee VALUES(1,'Joe',85000,1),(2,'Henry',80000,2),(3,'Sam',60000,2),(4,'Max',90000,1),(5,'Janet',69000,1),(6,'Randy',85000,1),(7,'Will',70000,1);\nCREATE SCHEMA active_users;\nCREATE TABLE active_users.Accounts(id INTEGER PRIMARY KEY,name TEXT);\nCREATE TABLE active_users.Logins(id INTEGER,login_date DATE);\nINSERT INTO active_users.Accounts VALUES(1,'Winston'),(7,'Jonathan');\nINSERT INTO active_users.Logins VALUES(7,'2020-05-30'),(1,'2020-05-30'),(7,'2020-05-31'),(7,'2020-06-01'),(7,'2020-06-02'),(7,'2020-06-02'),(7,'2020-06-03'),(1,'2020-06-07'),(7,'2020-06-10');\nSET search_path TO team_size;")
print("Example schemas and explicit source-matching fixtures ready.")

In [ ]:
%%sql example-0
SELECT employee_id, COUNT(*) OVER (PARTITION BY team_id) AS team_size
FROM team_size.Employee
ORDER BY employee_id;

In [ ]:
%%sql example-1
SELECT Department, Employee, Salary
FROM (
  SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary,
         DENSE_RANK() OVER (PARTITION BY d.id ORDER BY e.salary DESC) AS salary_rank
  FROM salary_ranking.Employee AS e
  JOIN salary_ranking.Department AS d ON e.departmentId = d.id
) AS ranked
WHERE salary_rank <= 3
ORDER BY Department, Salary DESC, Employee;

In [ ]:
%%sql example-2
WITH distinct_days AS (
  SELECT DISTINCT id, login_date FROM active_users.Logins
), login_windows AS (
  SELECT id, login_date,
         LAG(login_date, 4) OVER (PARTITION BY id ORDER BY login_date) AS lag4
  FROM distinct_days
)
SELECT DISTINCT w.id, a.name
FROM login_windows AS w
JOIN active_users.Accounts AS a ON a.id = w.id
WHERE w.login_date - w.lag4 = 4
ORDER BY w.id;

In [ ]:
from collections import Counter, defaultdict
from datetime import date,timedelta
import itertools
assert query_results['example-0'][1] == [(1,3),(2,3),(3,3),(4,1),(5,2),(6,2)]
salary_rows=connection.execute('SELECT d.name,e.name,e.salary FROM salary_ranking.Employee e JOIN salary_ranking.Department d ON e.departmentId=d.id').fetchall()
top={dept:sorted({salary for d,_,salary in salary_rows if d==dept},reverse=True)[:3]for dept,_,_ in salary_rows}
reference=sorted((dept,name,salary)for dept,name,salary in salary_rows if salary in top[dept])
assert sorted(query_results['example-1'][1])==reference
assert sum(row[0]=='IT'for row in reference)==4
assert query_results['example-2'][1]==[(7,'Jonathan')]
def active_reference(rows):
    days=set(rows)
    return any(all(day+timedelta(days=offset)in days for offset in range(5))for day in days)
login_query=query_texts['example-2']
# Exhaustive 7-day presence patterns, each with deliberately duplicated days.
for bits in itertools.product([False,True],repeat=7):
    days=[date(2020,1,1)+timedelta(days=i)for i,present in enumerate(bits)if present]
    connection.execute('TRUNCATE active_users.Logins')
    with connection.cursor()as cursor:
        cursor.executemany('INSERT INTO active_users.Logins VALUES(%s,%s)',[(7,d)for d in days for _ in range(2)])
    actual=connection.execute(login_query).fetchall()
    assert bool(actual)==active_reference(days),(bits,actual)
# Boundary across year/leap-day transitions and disjoint users.
for start in [date(2019,12,29),date(2020,2,27)]:
    days=[start+timedelta(days=i)for i in range(6)]
    connection.execute('TRUNCATE active_users.Logins')
    with connection.cursor()as cursor:cursor.executemany('INSERT INTO active_users.Logins VALUES(%s,%s)',[(7,d)for d in days]+[(1,start),(1,start+timedelta(days=4))])
    assert connection.execute(login_query).fetchall()==[(7,'Jonathan')]
print('All three original source datasets match expected results; independent salary ties and128 duplicate-login presence patterns plus calendar boundaries passed.')
connection.close()
server.cleanup()
assert not data_dir.exists()
print('Owned PostgreSQL server stopped; temporary database data removed.')

## Original source code and transcripts

These exact source blocks are preserved for comparison and are not executed.

### Original block 1

```text
SELECT 
  employee_id, 
  COUNT(employee_id) OVER (PARTITION BY team_id) AS team_size
FROM Employee
```

### Original block 2

```text
SELECT 
  Department, Employee, Salary
FROM (
  SELECT
    Employee.name AS Employee,
    Employee.Salary AS Salary,
    Department.name AS Department,
    DENSE_RANK() OVER (PARTITION BY Department.id 
                       ORDER BY Salary DESC) AS rank 
FROM Employee 
JOIN Department
ON Employee.departmentId = Departmentid
) AS t
WHERE rank <=3
```

### Original block 3

```text
SELECT 
DISTINCT t.id, a.name
FROM (
  SELECT
    id,
    login_date,
    LAG(login_date, 4) OVER(PARTITION BY id ORDER BY login_date) AS lag4
    
FROM logins
) t

JOIN Accounts a
ON a.id = t.id
WHERE DATEDIFF(day, lag4, login_date)=4;
```